In [165]:
import numpy as np
import pandas as pd
import neurokit2 as nk

Imported reshape function as it will be needed to make row-wise descriptives

In [166]:
#Now we make the reshape function
#4 parameters, your dataframe, your window size or how many hz of rows should be in each row, purity for how much each row can tolerate label confounsion, and stride size for how much we should overlap
#By default this is every 10 seconds of data
def reshape_function(df, name = 'eda', window_size = 60, stride_size = 15, hertz=8, purity_threshold=0.8):
    window_row = window_size*hertz
    stride_row = stride_size*hertz
    #Get the subject names
    subjects = sorted(
        list(set(col.split("_")[0] for col in df.columns if "_" in col))
    )
    #Store the chunks of data while we're looping across the time span
    all_window = []
    for i in subjects:
        signal_col = f'{i}_{name}'
        label_col = f'{i}_Label'
        #Kill NAs in individual sequence
        individual_df = df[[signal_col,label_col]].dropna()
        total_rows = len(individual_df)
        #For each subject we take a chunk of their data by our determined window size then stack onto a list (in this case "all window")
        #Chunk is determined through standard range check then we check the purity threshold if we keep the chunk, based on birken's paper design
        for j in range(0,total_rows-window_row+1, stride_row):
            end = j + window_row
            signal_window = individual_df[signal_col].iloc[j:end].values
            label_window = individual_df[label_col].iloc[j:end].values

            # Determine the majority label and calculate if there exists a supermajority in the chunk to determine if kept
            value_counts = pd.Series(label_window).value_counts()
            majority_label = value_counts.index[0]
            majority_count = value_counts.iloc[0]

            purity = majority_count / window_row

            # Retain only windows meeting or exceeding the requested purity ratio
            if purity >= purity_threshold:
                row_data = [i, majority_label] + list(signal_window)
                all_window.append(row_data)

    feature_cols = [f"sample_{i}" for i in range(window_row)]
    columns = ["subject_id", "label"] + feature_cols

    return pd.DataFrame(all_window, columns=columns)

    





I also want visualization boxplots for each feature set similar to Zhu lili's breakdown

Import tonic and phasic files

In [167]:
tonic = pd.read_csv("../Data/wesad_eda/tonic_signal.csv")
phasic = pd.read_csv("../Data/wesad_eda/phasic_signal.csv")

This section is just applying the reshape function and verifying the outputs

In [168]:
tonic_10 = reshape_function(tonic, name='Tonic')

In [169]:
tonic_10.shape

(2106, 482)

In [170]:
tonic_10.head()

,subject_id,label,sample_0,sample_1,sample_2,sample_3,sample_4,sample_5,sample_6,sample_7,...,sample_470,sample_471,sample_472,sample_473,sample_474,sample_475,sample_476,sample_477,sample_478,sample_479
0,S10,1.0,0.303524,0.302306,0.301094,0.299887,0.298688,0.297496,0.296313,0.295139,...,0.378796,0.378854,0.378911,0.378968,0.379025,0.379081,0.379137,0.379193,0.379248,0.379302
1,S10,1.0,0.311120,0.312091,0.313059,0.314024,0.314986,0.315946,0.316901,0.317854,...,0.375137,0.375090,0.375045,0.375003,0.374964,0.374928,0.374895,0.374865,0.374838,0.374815
2,S10,1.0,0.377588,0.377619,0.377646,0.377667,0.377684,0.377696,0.377705,0.377709,...,0.377661,0.377633,0.377604,0.377576,0.377547,0.377518,0.377490,0.377461,0.377433,0.377405
3,S10,1.0,0.377300,0.377304,0.377307,0.377309,0.377310,0.377310,0.377310,0.377308,...,0.386480,0.386649,0.386817,0.386985,0.387153,0.387321,0.387489,0.387656,0.387822,0.387989
4,S10,1.0,0.379356,0.379409,0.379461,0.379512,0.379562,0.379611,0.379660,0.379707,...,0.392576,0.392444,0.392311,0.392176,0.392040,0.391902,0.391763,0.391624,0.391483,0.391342


Tonic descriptives row-wise for max and mean

In [171]:
sample_cols = [col for col in tonic_10.columns if col.startswith('sample_')]
tonic_descriptives = pd.DataFrame()
#Bring Labels into descriptives to the databas
tonic_descriptives['label'] = tonic_10['label']
#Bring subject ID over to descriptives
if 'subject_id' in tonic_10.columns:
    tonic_descriptives['subject_id'] = tonic_10['subject_id']
#Descriptives
tonic_descriptives['tonic_mean'] = tonic_10[sample_cols].mean(axis=1)
tonic_descriptives['tonic_max']  = tonic_10[sample_cols].max(axis=1)

tonic_descriptives.head()

,label,subject_id,tonic_mean,tonic_max
0,1.0,S10,0.347428,0.379302
1,1.0,S10,0.371826,0.380334
2,1.0,S10,0.377376,0.380334
3,1.0,S10,0.378218,0.387989
4,1.0,S10,0.382165,0.395525


In [172]:
phasic_10 = reshape_function(phasic, name='Phasic')

Phasic descriptives row-wise for mean, peak counts and rise time standard deviation

In [173]:
phasic_10.shape

(2106, 482)

In [174]:
sample_cols = [col for col in phasic_10.columns if col.startswith('sample_')]
phasic_descriptives = pd.DataFrame()
#Bring Labels into descriptives to the database
phasic_descriptives['label'] = phasic_10['label']
#Bring subject ID over to descriptives
if 'subject_id' in phasic_10.columns:
    phasic_descriptives['subject_id'] = phasic_10['subject_id']
#Descriptives
phasic_descriptives['phasic_mean'] = phasic_10[sample_cols].mean(axis=1)
#phasic_descriptives['peaks_count'] = (phasic_10[sample_cols] > 0.01).sum(axis=1)
phasic_descriptives['phasic_skew'] = phasic_10[sample_cols].skew(axis=1)
phasic_descriptives.head()

,label,subject_id,phasic_mean,phasic_skew
0,1.0,S10,0.029969,11.144136
1,1.0,S10,0.003446,8.496083
2,1.0,S10,0.000575,16.284196
3,1.0,S10,0.001437,12.747561
4,1.0,S10,0.001248,14.075460


Test example before I implemented on full dataframe the rise time standard deviation descriptive

In [175]:
row_example = phasic_10.iloc[0:1,2:]
signals, info = nk.eda_peaks(np.array(row_example).flatten(), sampling_rate=8)
# Calculate rise times in seconds
rise_times_sec = (info["SCR_Peaks"] - info["SCR_Onsets"]) / 8
np.nanstd(rise_times_sec)


0.5315461619784773

In [176]:
def rise_time_std(row, sampling_rate = 8):
    try:
        #Uses neurokits peak detection to get the onsets then convert it to a time scale and calculate the standard deviation
        signals, info = nk.eda_peaks(row, sampling_rate=sampling_rate)
        # Calculate peak count
        peaks = info["SCR_Peaks"]
        peak_count = len(peaks)
        rise_times = (info["SCR_Peaks"] - info["SCR_Onsets"]) / sampling_rate
        rise_time_std = np.nanstd(rise_times)
        return pd.Series(
                    [peak_count, rise_time_std],
                    index=["phasic_peak_count", "phasic_std_rise_time"],
                )

    #There is a lot of rows with no peaks at all, need to decide how to handle the NAs where the std depreciates thereof
    except Exception:
        return np.nan
#Create a new column after apply row wise the above function
phasic_descriptives[
    ["phasic_peak_count", "phasic_std_rise_time"]
] = phasic_10[sample_cols].apply(rise_time_std, axis=1)

phasic_descriptives.head()

/home/sj/.local/lib/python3.12/site-packages/neurokit2/eda/eda_peaks.py:122: RuntimeWarning: All-NaN slice encountered
  valid_peaks = np.logical_and(info["SCR_Peaks"] > np.nanmin(info["SCR_Onsets"]), ~np.isnan(info["SCR_Onsets"]))  # pylint: disable=E1111
/home/sj/.local/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/sj/.local/lib/python3.12/site-packages/neurokit2/eda/eda_peaks.py:122: RuntimeWarning: All-NaN slice encountered
  valid_peaks = np.logical_and(info["SCR_Peaks"] > np.nanmin(info["SCR_Onsets"]), ~np.isnan(info["SCR_Onsets"]))  # pylint: disable=E1111
/home/sj/.local/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/sj/.local/lib/python3.12/site-packages/neurokit2/eda/eda_peaks.py:122: RuntimeWarning: All-NaN slice

,label,subject_id,phasic_mean,phasic_skew,phasic_peak_count,phasic_std_rise_time
0,1.0,S10,0.029969,11.144136,11.0,0.531546
1,1.0,S10,0.003446,8.496083,10.0,0.648479
2,1.0,S10,0.000575,16.284196,3.0,0.812500
3,1.0,S10,0.001437,12.747561,6.0,1.617483
4,1.0,S10,0.001248,14.075460,5.0,1.547983


Create and save file descriptive file

In [177]:
feature_dataframe = pd.concat(
    [phasic_descriptives, tonic_descriptives.drop(columns=["subject_id", "label"])],
    axis=1,
)
feature_dataframe.head()

,label,subject_id,phasic_mean,phasic_skew,phasic_peak_count,phasic_std_rise_time,tonic_mean,tonic_max
0,1.0,S10,0.029969,11.144136,11.0,0.531546,0.347428,0.379302
1,1.0,S10,0.003446,8.496083,10.0,0.648479,0.371826,0.380334
2,1.0,S10,0.000575,16.284196,3.0,0.812500,0.377376,0.380334
3,1.0,S10,0.001437,12.747561,6.0,1.617483,0.378218,0.387989
4,1.0,S10,0.001248,14.075460,5.0,1.547983,0.382165,0.395525


In [178]:
feature_dataframe.to_csv("../Data/wesad_eda/feature_dataframe.csv", index=False)
